## 머신러닝 기초

> **노트북 사용전 확인할 것**
> 
> 셀을 순서대로 실행하지 않으면 변수 상태가 꼬일 수 있음!\
> 위에서 아래 순서대로 실행하면서 확인하자!

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from _style import setup
from _ml import build_dataset, FEATURES

setup()
pd.set_option("display.width", 130)
pd.set_option("display.max_column", 20)

print("준비 완료")

준비 완료


### 1. 프로그래밍(코딩)과의 차이점

지금까지 우리가 작성해온 프로그래밍은 **규칙을 사람이 정하는 것**이었음!

```python
if 거래량 > 100_000 and 등락률 > 5:
    return "급등주"
```

규칙이 명확할 경우에는 프로그래밍이 빠르고, 정확하고 좋을 수 있음! (설명이 가능)

하지만, **규칙을 사람이 알수 없을 때**는 설명이 어려워짐!
> "오늘 주가가 오를 종목의 조건은 무엇인가?"

거래량, 등락률, 이동평균, ... 수십개 이상의 변수로 조건을 작성할 수 있겠지만, 어려움 있어
**머신러닝은 데이터에서 그 규칙을 컴퓨터가 찾아내게 하는 방법**이기 때문에 이 문제를 해소해 나갈 수 있음!

In [2]:
df = build_dataset()

# 규칙 : 거래량이 평소보다 2배 이상 많고, 20일 평균보다 3%이상 높으면 오를 것임!
rule = (df["volume_ratio"] >= 2.0) & (df["ma20_ratio"] >= 1.03)

# df["target_up"] -> 올랐으면 1, 아니면 0
hit = (rule == df["target_up"]).mean()

print(f" 규칙에 해당하는 날 : {rule.sum():,}건 / {len(df):,}건")
print(f" 규칙의 적중률 : {hit:.4f}")
print(f"전체 상승 비율: {df['target_up'].mean()}")

 규칙에 해당하는 날 : 731건 / 87,480건
 규칙의 적중률 : 0.5099
전체 상승 비율: 0.48954046639231824


> **적중률이 생각보다 낮게 나옴!**
>
> 사람이 느낌으로 만든 규칙이기 때문에, 어떤 조건을 어떤 값으로 잡아야하는지 어려움이 있음. 조건을 바꿔가면서 직접 찾을 수도 있지만 만약에 피처(변수)가 7개만 되어도, 조합이 폭발적으로 많아질 것임!

- 프로그래밍 : 데이터 + 규칙(사람)을 입력해서 결과를 받는다.
- 머신러닝 : 데이터 + 정답을 입력해서 규칙을 만든다.


> 규칙이 명확하면 직접 프로그래밍으로 답을 도출하고,\
> **규칙을 알 수 없을 때** 머신러닝을 이용해서 규칙을 만들자!

---
### 분류와 회귀

**무엇을 예측하려고 하는가?**

||분류|회귀|
|:--:|:--:|:--:|
|예측 가능한 것|범주|수치|
|질문|오를까?내릴까?|얼마나 오를까?|
|출력|up/down|2.7%|
|평가지표|정확도,정밀도,재현율|MAE,RMSE,...|

**동일한 데이터로 두 문제를 모두 도출할 수 있음!**

In [9]:
# target_ret : 수익률 - 수치 => 회귀
# target_up  : 올랐는지, 오르지 않았는지 (1/0) - 범주 => 분류

sample = df[["code", "date", "target_ret", "target_up"]].head(8)
print(sample.to_string(index=False))

print("같은 행이지만 정답을 두 가지로 확인할 수 있음! (회귀/분류)")

 code       date  target_ret  target_up
G0001 2023-10-24    0.015574          1
G0001 2023-10-25   -0.008639          0
G0001 2023-10-26   -0.002960          0
G0001 2023-10-27    0.002676          1
G0001 2023-10-30    0.003128          1
G0001 2023-10-31    0.046732          1
G0001 2023-11-01   -0.001827          0
G0001 2023-11-02    0.011819          1
같은 행이지만 정답을 두 가지로 확인할 수 있음! (회귀/분류)


> **평가 지표는 섞어서 사용할 수 없음**
>
> 분류 모델에 MAE, RMSE, ... 을 사용하거나 \
> 회귀 모델에 정확도를 사용할 수 있을까요..? 사용할 수 없음! \
> "정확도 92%" 라는 판단은 **분류에서만** 의미가 있으므로, 회귀에서는 "정확하게 맞췄다"는 개념 자체가 성립되지 않는다.